In [1]:
# tests/smoke/test_biblioteca_smoke.py
import pytest
import requests

BASE = 'http://localhost:8000/api'

@pytest.mark.smoke
class TestBibliotecaSmoke:

    def test_01_api_disponible(self):
        res = requests.get(f"{BASE}/health")
        assert res.status_code == 200
        assert res.json()["status"] == "ok"

    def test_02_login_funciona(self):
        data = {
            "username": "test_user",
            "password": "123456"
        }
        res = requests.post(f"{BASE}/auth/login", json=data)
        assert res.status_code == 200
        assert "access_token" in res.json()

    def test_03_listar_libros_responde(self):
        res = requests.get(f"{BASE}/libros")
        assert res.status_code == 200
        assert isinstance(res.json(), list)

    def test_04_base_de_datos_conectada(self):
        res = requests.get(f"{BASE}/health")
        assert res.status_code == 200
        assert res.json()["db"] == "connected"

    def test_05_modulo_multas_responde(self):
        estudiante_id = 1
        res = requests.get(f"{BASE}/multas/{estudiante_id}")
        assert res.status_code == 200
        assert "multa_total" in res.json()

In [7]:
!pip install pytest

In [13]:
import sys
import os

sys.path.append(os.getcwd())

In [14]:
import importlib
import biblioteca
importlib.reload(biblioteca)

<module 'biblioteca' from '/content/biblioteca/__init__.py'>

In [8]:
import os

os.makedirs("biblioteca", exist_ok=True)
os.makedirs("tests/smoke", exist_ok=True)
os.makedirs("tests/regression", exist_ok=True)

open("biblioteca/__init__.py", "w").close()

In [9]:
%%writefile biblioteca/prestamos.py

from datetime import datetime

libros_prestados = {}

def registrar_prestamo(libro_id, fecha=None):
    libros_prestados[libro_id] = fecha or datetime.now()

def registrar_devolucion(libro_id):
    if libro_id in libros_prestados:
        del libros_prestados[libro_id]

def calcular_disponibilidad(libro_id):
    return libro_id not in libros_prestados

Writing biblioteca/prestamos.py


In [10]:
%%writefile biblioteca/multas.py

def calcular_multa(dias_retraso):
    tarifa = 500
    return dias_retraso * tarifa

Writing biblioteca/multas.py


In [11]:
%%writefile tests/regression/test_prestamos_regression.py

import pytest
from datetime import datetime, timedelta

from biblioteca.prestamos import calcular_disponibilidad
from biblioteca.prestamos import registrar_prestamo, registrar_devolucion
from biblioteca.multas import calcular_multa


@pytest.mark.regression
class TestPrestamosRegression:

    def setup_method(self):
        # limpiar estado antes de cada test
        from biblioteca.prestamos import libros_prestados
        libros_prestados.clear()

    def test_libro_prestado_no_esta_disponible(self):
        libro_id = 1
        registrar_prestamo(libro_id)

        assert calcular_disponibilidad(libro_id) is False

    def test_libro_disponible_tras_devolucion(self):
        libro_id = 2
        registrar_prestamo(libro_id)
        registrar_devolucion(libro_id)

        assert calcular_disponibilidad(libro_id) is True

    def test_bug_2024_031_no_disponible_con_prestamo_antiguo(self):
        libro_id = 3
        fecha_antigua = datetime.now() - timedelta(days=45)

        registrar_prestamo(libro_id, fecha=fecha_antigua)

        assert calcular_disponibilidad(libro_id) is False

    def test_multa_no_afecta_disponibilidad_de_otro_libro(self):
        libro1 = 4
        libro2 = 5

        registrar_prestamo(libro1)

        assert calcular_disponibilidad(libro2) is True

    def test_calculo_multa_tres_dias_retraso(self):
        assert calcular_multa(3) == 1500

Writing tests/regression/test_prestamos_regression.py


In [16]:
!ls -R

.:
biblioteca  sample_data  tests

./biblioteca:
__init__.py  multas.py	prestamos.py  __pycache__

./biblioteca/__pycache__:
__init__.cpython-312.pyc

./sample_data:
anscombe.json		      mnist_test.csv
california_housing_test.csv   mnist_train_small.csv
california_housing_train.csv  README.md

./tests:
regression  smoke

./tests/regression:
__pycache__  test_prestamos_regression.py

./tests/regression/__pycache__:
test_prestamos_regression.cpython-312-pytest-8.4.2.pyc

./tests/smoke:


In [20]:
%%writefile pytest.ini
[pytest]
markers =
    regression: tests de regresión
    smoke: tests básicos del sistema

Writing pytest.ini


In [22]:
!python -m pytest tests/regression -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
configfile: pytest.ini
plugins: typeguard-4.5.1, anyio-4.13.0, langsmith-0.7.30
collected 5 items                                                              

tests/regression/test_prestamos_regression.py::TestPrestamosRegression::test_libro_prestado_no_esta_disponible PASSED [ 20%]
tests/regression/test_prestamos_regression.py::TestPrestamosRegression::test_libro_disponible_tras_devolucion PASSED [ 40%]
tests/regression/test_prestamos_regression.py::TestPrestamosRegression::test_bug_2024_031_no_disponible_con_prestamo_antiguo PASSED [ 60%]
tests/regression/test_prestamos_regression.py::TestPrestamosRegression::test_multa_no_afecta_disponibilidad_de_otro_libro PASSED [ 80%]
tests/regression/test_prestamos_regression.py::TestPrestamosRegression::test_calculo_multa_tres_dias_retraso PA